# Sicilian–English–Italian Neural Machine Translation
### A reproducible, end-to-end pipeline

This notebook reproduces our whole model from start to finish, structured like a short
paper. Each section explains *what* and *why* (as in the slides), then runs the code — the
implementations live in the repo's `.py` files (`experiments/nllb/nllb_pipeline.py`,
`experiments/dataset/normalize_scn.py`), the notebook only orchestrates.

**Runtime → Change runtime type → GPU (A100 for §7.2–§7.4).** Every cell that trains skips the
training whose adapter is already on Drive, and every cell that evaluates prints the stored
results file when it exists, so *Run all* computes only what is missing and reprints the rest.

## 1. Introduction

Sicilian is a low-resource Romance language. Wdowiak's *Tradutturi Sicilianu* trains small
Transformers from scratch; his reverse-training recipe builds a custom pretrained model in stages
from synthetic and web text before fine-tuning on the literary text of *Arba Sicula*.

This notebook runs the experiments of our paper in its order: the corpus and its normalization
(§3–§4), NLLB-200 1.3B zero-shot and with a bidirectional LoRA adapter on the web set (§5–§6),
back-translation, the staged training on Wdowiak's back-translations and on the curated set, the
curriculum and the ablation that chooses the final model (§7.1–§7.4), the Italian directions
(§7.5–§7.7) and the comparison on Wdowiak's own test set (§7.8). The from-scratch Sockeye models
and the retraining of Wdowiak's 2022 baseline are in `experiments/baseline/recipe_colab.ipynb`.

## 2. Setup
Clone the repo for the `.py` code, install dependencies, mount Drive.

In [47]:
# transformers 5 removed a method that the peft preinstalled on Colab still calls, so the
# pair is pinned to versions that work together
!pip -q install transformers==4.57.6 peft==0.17.1 datasets sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
sys.path += ['SicilianNMT/experiments/nllb', 'SicilianNMT/experiments/dataset']
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'; DATA = f'{OUT}/data'
def read(p): return open(p, encoding='utf-8').read().splitlines()
from nllb_pipeline import load_base, attach_lora, build_dataset, finetune, translate, score
from normalize_scn import normalize
print('ready')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready


## 3. Data

### 3.1 Sources, extraction & alignment
The parallel text is built automatically: extract the *Arba Sicula* PDFs with PyMuPDF,
classify each page's language, confirm facing pairs and align sentences with **LaBSE**
embeddings + a monotonic dynamic program (no dictionary, no hunalign). We then add the
curated Napizia NLLB / WikiMatrix datasets and deduplicate. Code:
`experiments/extraction/build_all.py` and `experiments/dataset/assemble.py` (they require the
PDFs and the public datasets). A 1k/1k valid/test split is **frozen** so every result stays
comparable.

### 3.2 Loading the prepared dataset

In [48]:
train = {l: read(f'{DATA}/train.{l}') for l in ('scn', 'en')}
test  = {l: read(f'{DATA}/test.{l}')  for l in ('scn', 'en')}
print('train', len(train['scn']), '| test', len(test['scn']))
print('sample pair:\n  scn:', train['scn'][0], '\n  en :', train['en'][0])

train 27386 | test 1000
sample pair:
  scn: Esistunu na para di libbra di grammatica ma sunnu opiri di consultazioni. 
  en : There are several grammars, but they are primarily reference books, not teach­ ing textbooks.


## 4. Preprocessing

### 4.1 Standardization (Standard Sicilian)
Sicilian has no single official orthography (many spellings, omitted accents). We normalize
toward **Standard Sicilian** — Cipolla's standard, augmented by Wdowiak for MT clarity — with
`normalize_scn.py`. The light **std** level applies spelling rules (*cchiù→chiù*,
*io/iu/eu→jo*, *non→nun*, *çiuri→ciuri*) and keeps accents/contractions; the **full** level
additionally uncontracts (*dû→di lu*) and ASCII-folds. We use **std** (its effect is measured
in §4.2).

In [49]:
print('before:', test['scn'][0])
for d in (train, test):
    d['scn'] = [normalize(s, 'std') for s in d['scn']]
print('after :', test['scn'][0])

before: Nun eppimu mai lu piaciri di vidiri i so opiri ma sta pasta di sicuru fa arrisbigliari i papiddi gustativi da lingua di tutti, siciliani e no.
after : Nun eppimu mai lu piaciri di vidiri i so opiri ma sta pasta di sicuru fa arrisbigliari i papiddi gustativi da lingua di tutti, siciliani e no.


### 4.2 Does normalization help NLLB?
A controlled comparison: three arms (**raw / std / full**), scn→en only, a fresh model each,
**identical English references** (so BLEU is directly comparable). Variants are generated
in-notebook by `normalize_scn`. Results go to `results_normalization.json`.

In [50]:
# --- 4.2 normalization ablation (raw / std / full), scn->en ---
p = f'{OUT}/results_normalization.json'
if not os.path.exists(p):
    raw = {s: read(f'{DATA}/{s}.scn') for s in ('train', 'test')}   # un-normalized
    abl = {}
    for arm in ('raw', 'std', 'full'):
        f = (lambda x: x) if arm == 'raw' else (lambda x, a=arm: normalize(x, a))
        m, tok = load_base('facebook/nllb-200-1.3B'); aft = attach_lora(m)
        aft = finetune(aft, tok, build_dataset(tok, [([f(x) for x in raw['train']], train['en'], 'scn', 'en')]),
                       out_dir=f'{OUT}/abl-{arm}', epochs=1)
        abl[arm] = score(translate(aft, tok, [f(x) for x in raw['test']], 'scn', 'en'), test['en'])
    json.dump(abl, open(p, 'w'), indent=2)
for arm, (b, c) in json.load(open(p)).items():
    print(f'{arm}:  BLEU={b}  chrF={c}')

raw:  BLEU=30.67  chrF=56.31
std:  BLEU=30.79  chrF=56.49
full:  BLEU=30.61  chrF=56.36


### 4.3 Tokenization & subwords
NLLB-200 carries its own SentencePiece tokenizer, so the model side needs no extra
tokenization. For the from-scratch Sockeye baseline we instead port the Napizia tokenizer
(ASCII-fold + uncontract) and bias a small joint BPE toward textbook **desinences**
(*Carinisi are dogs!* → `car@@ in@@ isi are dogs !`), cutting the vocabulary 14k→2k — see
`experiments/tokenization/` and `experiments/baseline/`.

## 5. Model & training

### 5.1 NLLB-200 + LoRA
NLLB-200 is a massively multilingual encoder–decoder that **supports Sicilian** (`scn_Latn`) —
the equivalent of the costly pretraining a from-scratch system would need. We adapt it with
**LoRA**: freeze the weights $W_0$ and learn a low-rank update $W=W_0+\frac{\alpha}{r}BA$
($r{=}32$) on the attention projections — ~1.3% of the parameters, fits a T4.

### 5.2 Zero-shot NLLB-200 1.3B
The base model without any adapter, on raw (un-normalized) text: scn↔en on our test set and
it↔scn on the WikiMatrix test set (the zero-shot rows of Tables 2 and 4 of the paper). Decoding
only; results to `results_zeroshot.json`.

In [51]:
# --- 5.2 zero-shot NLLB-200 1.3B on raw text ---
p = f'{OUT}/results_zeroshot.json'
if not os.path.exists(p):
    raw_test = {'scn': read(f'{DATA}/test.scn'), 'en': read(f'{DATA}/test.en')}
    raw_isc = {l: read(f'{DATA}/itsc_test.{l}') for l in ('it', 'scn')}
    m, tok = load_base('facebook/nllb-200-1.3B')
    zs = {f'{s}->{t}': score(translate(m, tok, te[s], s, t), te[t])
          for s, t, te in [('scn', 'en', raw_test), ('en', 'scn', raw_test), ('it', 'scn', raw_isc), ('scn', 'it', raw_isc)]}
    json.dump(zs, open(p, 'w'), indent=2)
for k, (b, c) in json.load(open(p)).items():
    print(f'zero-shot {k}:  BLEU={b}  chrF={c}')

zero-shot scn->en:  BLEU=29.0  chrF=55.23
zero-shot en->scn:  BLEU=9.9  chrF=41.09
zero-shot it->scn:  BLEU=17.57  chrF=44.76
zero-shot scn->it:  BLEU=42.23  chrF=59.05


### 5.3 Bidirectional fine-tune (scn↔en)
We train **both directions at once** so one adapter serves both and the weak reverse
direction is rescued (each example is tagged with its forced target language), for two epochs
on the web set. The adapter goes to `nllb-lora-bidir-1.3B`.

In [52]:
# --- 5.3 bidirectional LoRA adapter on the web set ---
BIDIR = f'{OUT}/nllb-lora-bidir-1.3B'
if not os.path.exists(f'{BIDIR}/adapter_config.json'):
    model, tok = load_base('facebook/nllb-200-1.3B'); ft = attach_lora(model)
    ds = build_dataset(tok, [(train['scn'], train['en'], 'scn', 'en'),
                             (train['en'], train['scn'], 'en', 'scn')])
    ft = finetune(ft, tok, ds, out_dir=BIDIR, epochs=2)
print('adapter:', BIDIR)

adapter: /content/drive/MyDrive/SicilianNMT-colab/nllb-lora-bidir-1.3B


## 6. Evaluation (test)

### 6.1 Metrics
We score on the **frozen** literary test set with **sacreBLEU** (BLEU, `tok:13a`) and
**chrF** — chrF being robust for a morphology-rich language. We report both directions.

### 6.2 Main result

In [53]:
# --- 6.2 the bidirectional adapter on our test set ---
p = f'{OUT}/results_reproduce.json'
if not os.path.exists(p):
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=BIDIR)
    results = {f'{s}->{t}': score(translate(m, tok, test[s], s, t), test[t]) for s, t in [('scn', 'en'), ('en', 'scn')]}
    json.dump(results, open(p, 'w'), indent=2)
for k, (b, c) in json.load(open(p)).items():
    print(f'{k}:  BLEU={b}  chrF={c}')

scn->en:  BLEU=31.27  chrF=56.92
en->scn:  BLEU=19.19  chrF=50.02


## 7. Further experiments
We now apply the feasible parts of Wdowiak's recipe on top of NLLB, in the order of the paper.
Each experiment reloads a fresh model; the heavy ones need Colab Pro.

### 7.1 Back-translation
Back-translate in-domain monolingual English (`mono_en.txt`, harvested from unaligned text)
en→scn with the bidirectional adapter to make synthetic Sicilian, then fine-tune scn→en on
real + synthetic for one epoch, on top of the bidirectional adapter. Results to `results_bt.json`.

In [54]:
# --- 7.1 back-translation of the unaligned English ---
p = f'{OUT}/results_bt.json'
if not os.path.exists(p):
    mono = read(f'{DATA}/mono_en.txt')[:15000]
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=BIDIR)
    synth = translate(m, tok, mono, 'en', 'scn', bs=16, beams=4)
    syn, seen = [], set(train['scn'])
    for sc, en in zip(synth, mono):
        sc, en = sc.strip(), en.strip()
        if sc and en and sc not in seen and 0.5 <= len(sc.split())/max(1, len(en.split())) <= 2.0:
            seen.add(sc); syn.append((sc, en))
    print(f'kept {len(syn)} synthetic pairs')
    bft = attach_lora(m)
    bft = finetune(bft, tok, build_dataset(tok, [(train['scn'] + [s for s, _ in syn],
                   train['en'] + [e for _, e in syn], 'scn', 'en')]),
                   out_dir=f'{OUT}/nllb-lora-bt-1.3B', epochs=1)
    b, c = score(translate(bft, tok, test['scn'], 'scn', 'en'), test['en'])
    json.dump({'BT scn->en 1.3B': {'bleu': b, 'chrf': c}}, open(p, 'w'), indent=2)
r = json.load(open(p))['BT scn->en 1.3B']
print(f"BT scn->en:  BLEU={r['bleu']}  chrF={r['chrf']}")

BT scn->en:  BLEU=31.6  chrF=57.21


### 7.2 Staged training: Wdowiak's back-translations, then the curated set (Colab Pro)
The complete pipeline on our NLLB+LoRA stack, using Eryk's delivered data (all PRIVATE, on
`OUT/eryk/`). Stage 2: fine-tune NLLB-1.3B on the forward-scored back-translations (the K lowest-scoring
pairs per direction, `<unk>` lines, near-copies and the `<2xxx>` tag dropped) across scn/en/it + the natural WikiMatrix it-en. Stage 3:
continue on the high-quality combined set (`OUT/finetune/`), leakage-guarded, eval in std space.
NLLB replaces his Sockeye stages 1-2, so the se36c/se37a checkpoints are not used here.
Results to `results_full.json`.

In [55]:
# --- 7.2 staged training: back-translations (stage 2), then the curated set (stage 3) ---
import math
import re
from difflib import SequenceMatcher
from normalize_scn import normalize
BT = f'{OUT}/eryk'; FT = f'{OUT}/finetune'
TOPK = 100000                      # per-direction cap on back-translations (raise on A100)
COPY = 0.9                         # drop near-copies: the forward model scores copying as easy
MINW = 5                           # drop short web fragments (titles, menus), they score low too
_TAG = re.compile(r'^<2[a-z]{3}>\s*')
_W = re.compile(r"[a-zàèéìòùâêîôû']+")
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')], 'en': read(f'{DATA}/test.en')}
ban = set(test['scn']) | set(read(f'{DATA}/test.scn')) | set(test['en'])   # leakage guard, both stages


p = f'{OUT}/results_full.json'
if not os.path.exists(p):
    def sicilianish(t):
        """Italian leaks into the low-loss end: at most 1 word in -o, and no more than words in -u."""
        ws = [w for w in _W.findall(t.lower()) if len(w) >= 3]
        o = sum(w.endswith(('o', 'ò')) for w in ws)
        return o <= 1 and o <= sum(w.endswith(('u', 'ù')) for w in ws)

    def load_scored(fname, src, tgt, k):
        # Column 1 is Sockeye's negative log probability that the target is a translation of
        # the source, so a low score is a good score (E. Wdowiak, 2026-09-16): exp(-score) is
        # the probability, 0.10 -> 90.5%, 1.00 -> 36.8%, 3.00 -> 5.0%. Take the k lowest, and
        # drop what scores low for the wrong reason: <unk> lines, short web fragments,
        # near-copies of the source, and Italian passing as Sicilian.
        rows = []
        for line in read(f'{BT}/{fname}'):
            if '<unk>' in line: continue
            p = line.split('\t')
            if len(p) < 3: continue
            try: sc = float(p[0])
            except ValueError: continue
            s = _TAG.sub('', p[1]).strip(); t = p[2].strip()
            if s and t and s != t and s not in ban and t not in ban: rows.append((sc, s, t))
        rows.sort(key=lambda r: r[0])
        kept = []
        for sc, s, t in rows:
            if len(s.split()) < MINW or len(t.split()) < MINW: continue
            if not sicilianish(t if tgt == 'scn' else s if src == 'scn' else 'u'): continue
            sm = SequenceMatcher(None, s.lower(), t.lower())
            if sm.quick_ratio() >= COPY and sm.ratio() >= COPY: continue
            kept.append((sc, s, t))
            if len(kept) == k: break
        cut = kept[-1][0] if kept else float('nan')
        print(f'  {fname}: kept {len(kept)} ({src}->{tgt}), '
              f'worst kept score {cut:.3f} = p {100 * math.exp(-cut):.1f}%')
        return ([r[1] for r in kept], [r[2] for r in kept], src, tgt)

    stage2 = [
        load_scored('nllb-as-mono-eng_bt-s2e.scores.txt', 'scn', 'en', TOPK),
        load_scored('nllb-as-mono-scn_bt-e2s.scores.txt', 'en', 'scn', TOPK),
        load_scored('nllb-as-mono-scn_bt-i2s.scores.txt', 'it', 'scn', TOPK),
        load_scored('wm-million-as-mono-ita_bt-s2i.scores.txt', 'scn', 'it', TOPK),
        load_scored('as-mono-scn_bt-e2s.scores.txt', 'en', 'scn', TOPK),
        load_scored('as-mono-scn_bt-i2s.scores.txt', 'it', 'scn', TOPK),
    ]
    wm = [(e, i) for e, i in zip(read(f'{BT}/wm-million_tkn2ascii.en-it.txt.en'),
                                  read(f'{BT}/wm-million_tkn2ascii.en-it.txt.it')) if e not in ban][:TOPK]
    wm_en = [e for e, _ in wm]; wm_it = [i for _, i in wm]
    stage2 += [(wm_en, wm_it, 'en', 'it'), (wm_it, wm_en, 'it', 'en')]

    m, tok = load_base('facebook/nllb-200-1.3B'); ad = attach_lora(m)
    ad = finetune(ad, tok, build_dataset(tok, stage2),
                  out_dir=f'{OUT}/nllb-lora-stage2-1.3B', epochs=1, batch_size=16, grad_accum=1)

    # stage 3: combined high-quality fine-tune (leakage-guarded, std-space eval)
    tr = {'scn': read(f'{FT}/train.scn-en.scn'), 'en': read(f'{FT}/train.scn-en.en')}
    ti = {'scn': read(f'{FT}/train.scn-it.scn'), 'it': read(f'{FT}/train.scn-it.it')}
    se = [(s, e) for s, e in zip(tr['scn'], tr['en']) if s not in ban and e not in ban]
    ds3 = build_dataset(tok, [([s for s, _ in se], [e for _, e in se], 'scn', 'en'),
                              ([e for _, e in se], [s for s, _ in se], 'en', 'scn'),
                              (ti['scn'], ti['it'], 'scn', 'it'),
                              (ti['it'], ti['scn'], 'it', 'scn')])
    ad = finetune(ad, tok, ds3, out_dir=f'{OUT}/nllb-lora-full-1.3B', epochs=2, batch_size=16, grad_accum=1)

    res = {}
    for s, t in [('scn', 'en'), ('en', 'scn')]:
        res[f'{s}->{t}'] = score(translate(ad, tok, test[s], s, t), test[t])
        print(f'{s}->{t}:  BLEU={res[f"{s}->{t}"][0]}  chrF={res[f"{s}->{t}"][1]}')
    json.dump(res, open(f'{OUT}/results_full.json', 'w'), indent=2)
for k, (b, c) in json.load(open(p)).items():
    print(f'{k}:  BLEU={b}  chrF={c}')

scn->en:  BLEU=32.4  chrF=58.19
en->scn:  BLEU=21.72  chrF=52.79


### 7.3 Curriculum: use all the data, worst first (Colab Pro)
Eryk's proposal (2026-09-18): rather than discarding the back-translations that score badly,
spend them in a first, coarse stage, *"kind of like a fat hammer, bludgeoning and spreading
the output probabilities across dialects, leaving dull peaks in approximately the right
locations"*, and let the later stages sharpen those peaks, first with the good
back-translations (§7.2 stage 2), then with the curated set (§7.2 stage 3). Same data, same
test set and same stages as §7.2, with one stage prepended, so the two runs are comparable:
if the curriculum helps, `results_curriculum.json` beats `results_full.json`.

The coarse stage takes the WORST-scoring pairs per direction (`KBAD`), keeping the Italian
and other-dialect text that §7.2 filters out on purpose, since that spread is the point here;
`<unk>` lines, near-copies and anything touching the test set stay out of both stages. **Run
§7.2 first**: without its numbers this cell measures nothing.

Each stage checkpoints on Drive and resumes after a disconnection, and a stage whose adapter
is already saved is skipped, so the run can span several Colab sessions.

In [56]:
# --- 7.3 curriculum: coarse stage on the badly scored back-translations, then the stages of 7.2 ---
!git -C SicilianNMT pull -q
import importlib, nllb_pipeline; importlib.reload(nllb_pipeline)
from nllb_pipeline import load_base, attach_lora, build_dataset, finetune, translate, score
import math
import re
from difflib import SequenceMatcher
from normalize_scn import normalize
BT = f'{OUT}/eryk'; FT = f'{OUT}/finetune'
KBAD, KGOOD = 200000, 100000       # per-direction caps: coarse stage, then the good stage
COPY = 0.9                         # drop near-copies in both stages (they teach copying)
MINW, MINW_BAD = 5, 3              # the coarse stage may be noisy, but not menus and titles
_TAG = re.compile(r'^<2[a-z]{3}>\s*')
_W = re.compile(r"[a-zàèéìòùâêîôû']+")
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')], 'en': read(f'{DATA}/test.en')}
ban = set(test['scn']) | set(read(f'{DATA}/test.scn')) | set(test['en'])
FILES = [('nllb-as-mono-eng_bt-s2e.scores.txt', 'scn', 'en'),
         ('nllb-as-mono-scn_bt-e2s.scores.txt', 'en', 'scn'),
         ('nllb-as-mono-scn_bt-i2s.scores.txt', 'it', 'scn'),
         ('wm-million-as-mono-ita_bt-s2i.scores.txt', 'scn', 'it'),
         ('as-mono-scn_bt-e2s.scores.txt', 'en', 'scn'),
         ('as-mono-scn_bt-i2s.scores.txt', 'it', 'scn')]

def sicilianish(t):
    """Italian leaks into the low-score end: at most 1 word in -o, and no more than words in -u."""
    ws = [w for w in _W.findall(t.lower()) if len(w) >= 3]
    o = sum(w.endswith(('o', 'ò')) for w in ws)
    return o <= 1 and o <= sum(w.endswith(('u', 'ù')) for w in ws)

def load_band(fname, src, tgt, k, coarse):
    """k pairs from one score file: the worst-scoring ones if `coarse`, else the selection of §7.2."""
    rows = []
    for line in read(f'{BT}/{fname}'):
        if '<unk>' in line: continue
        p = line.split('\t')
        if len(p) < 3: continue
        try: sc = float(p[0])
        except ValueError: continue
        s = _TAG.sub('', p[1]).strip(); t = p[2].strip()
        if not s or not t or s == t or s in ban or t in ban: continue
        if len(s.split()) < (MINW_BAD if coarse else MINW): continue
        if len(t.split()) < (MINW_BAD if coarse else MINW): continue
        if not coarse and not sicilianish(t if tgt == 'scn' else s if src == 'scn' else 'u'): continue
        rows.append((sc, s, t))
    rows.sort(key=lambda r: r[0], reverse=coarse)
    kept = []
    for r in rows:
        sm = SequenceMatcher(None, r[1].lower(), r[2].lower())
        if sm.quick_ratio() < COPY or sm.ratio() < COPY:
            kept.append(r)
            if len(kept) == k: break
    lo, hi = kept[0][0], kept[-1][0]
    print(f'  {fname}: {len(kept)} pairs ({src}->{tgt}), scores {lo:.2f} to {hi:.2f} '
          f'= p {100 * math.exp(-lo):.1f}% to {100 * math.exp(-hi):.1f}%')
    return ([r[1] for r in kept], [r[2] for r in kept], src, tgt)

p = f'{OUT}/results_curriculum.json'
if os.path.exists(p):
    for k, (b, c) in json.load(open(p)).items():
        print(f'{k}:  BLEU={b}  chrF={c}')
else:
    # Each stage saves a checkpoint every 2000 steps and resumes from it, and a stage whose adapter
    # is already on Drive is not trained again (the coarse stage alone takes about 11 h on an A100,
    # more than one Colab session).
    COARSE, GOOD = f'{OUT}/nllb-lora-coarse-1.3B', f'{OUT}/nllb-lora-cur-stage2-1.3B'
    done = lambda d: os.path.exists(f'{d}/adapter_config.json')
    m, tok = load_base('facebook/nllb-200-1.3B')
    if done(GOOD):
        ad = attach_lora(m, adapter=GOOD)
    else:
        if done(COARSE):
            ad = attach_lora(m, adapter=COARSE)
        else:
            print('coarse stage (worst scores):')
            stage_bad = [load_band(f, s, t, KBAD, True) for f, s, t in FILES]
            ad = finetune(attach_lora(m), tok, build_dataset(tok, stage_bad), out_dir=COARSE,
                          epochs=1, batch_size=16, grad_accum=1, save='steps')
        print('good stage (best scores):')
        stage_good = [load_band(f, s, t, KGOOD, False) for f, s, t in FILES]
        wm = [(e, i) for e, i in zip(read(f'{BT}/wm-million_tkn2ascii.en-it.txt.en'),
                                     read(f'{BT}/wm-million_tkn2ascii.en-it.txt.it')) if e not in ban][:KGOOD]
        wm_en = [e for e, _ in wm]; wm_it = [i for _, i in wm]
        stage_good += [(wm_en, wm_it, 'en', 'it'), (wm_it, wm_en, 'it', 'en')]
        ad = finetune(ad, tok, build_dataset(tok, stage_good), out_dir=GOOD,
                      epochs=1, batch_size=16, grad_accum=1, save='steps')

    # final stage: the curated set, exactly as in 7.2 (leakage-guarded, std-space eval)
    tr = {'scn': read(f'{FT}/train.scn-en.scn'), 'en': read(f'{FT}/train.scn-en.en')}
    ti = {'scn': read(f'{FT}/train.scn-it.scn'), 'it': read(f'{FT}/train.scn-it.it')}
    se = [(s, e) for s, e in zip(tr['scn'], tr['en']) if s not in ban and e not in ban]
    ds3 = build_dataset(tok, [([s for s, _ in se], [e for _, e in se], 'scn', 'en'),
                              ([e for _, e in se], [s for s, _ in se], 'en', 'scn'),
                              (ti['scn'], ti['it'], 'scn', 'it'),
                              (ti['it'], ti['scn'], 'it', 'scn')])
    ad = finetune(ad, tok, ds3, out_dir=f'{OUT}/nllb-lora-curriculum-1.3B',
                  epochs=2, batch_size=16, grad_accum=1, save='steps')

    res = {}
    for s, t in [('scn', 'en'), ('en', 'scn')]:
        res[f'{s}->{t}'] = score(translate(ad, tok, test[s], s, t), test[t])
    json.dump(res, open(f'{OUT}/results_curriculum.json', 'w'), indent=2)
    try: base = json.load(open(f'{OUT}/results_full.json'))
    except FileNotFoundError: base = {}
    for k, v in res.items():
        b = base.get(k)
        delta = f'  (7.2: BLEU={b[0]} chrF={b[1]}, {v[0] - b[0]:+.2f} BLEU)' if b else ''
        print(f'{k}:  BLEU={v[0]}  chrF={v[1]}{delta}')

scn->en:  BLEU=33.01  chrF=58.62
en->scn:  BLEU=21.78  chrF=52.94


### 7.4 Ablation and final model (Colab Pro, A100)
§7.2 with one change at a time, so that each lever is measured on its own and the best
model is chosen on evidence. The variants are cumulative: each keeps the changes of the one
before and adds one, i.e., (i) `base`, the settings of §7.2, (ii) three times the
back-translations in stage 2 (`topk` 300k per file), (iii) a larger adapter (rank 64, also on
the feed-forward layers), and (iv) longer sequences (256 tokens instead of 128, which cuts
about 2% of the curated pairs) with up to three epochs in stage 3.
A further variant, (v), asks whether keeping it$\leftrightarrow$en in stage 3 limits the
forgetting of those directions and helps the Sicilian ones: it retrains only stage 3 of (iv),
from the same stage-2 adapter, with 10,427 natural WikiMatrix it-en pairs per direction that
stage 2 did not use.
The final model, (vi), uses all the data: (iv) plus the 9,389 WikiMatrix it-scn training pairs of
the web set in stage 3 (their valid and test splits stay out), with up to four epochs.

In every variant the epoch of stage 3 is chosen on the 1,000-pair **validation** set, which
is held out of both stages, and only that epoch is scored on the test set; each stage
checkpoints on Drive and resumes after a disconnection, and a variant whose
`results_<name>.json` exists is skipped. `base` also checks the reproducibility of §7.2:
it differs only by the validation guard, so its test score should agree with
`results_full.json` within the run-to-run noise. The code is `experiments/nllb/staged.py`.
**Long run:** (ii) to (iv) have three times the stage-2 data of §7.2; run the variants
one per session if needed, and uncomment the 3.3B one only if (iv) still improves.

In [57]:
# --- 7.4 ablation of the staged training (see experiments/nllb/staged.py) ---
!git -C SicilianNMT pull -q
import importlib, nllb_pipeline, staged
importlib.reload(nllb_pipeline); importlib.reload(staged)
from nllb_pipeline import ATTN_FFN
BIG = dict(topk=300000)
BIG_FFN = dict(BIG, r=64, alpha=128, targets=ATTN_FFN)
LONG = dict(BIG_FFN, max_len=256, dec_len=256, stage3_epochs=3)
# cumulative variants; the strongest one runs right after base, so that the best model is known
# early, and the two intermediate ones complete the ablation afterwards
VARIANTS = [('base', {}),              # the settings of §7.2, epoch chosen on validation
            ('bt300-ffn64-long', LONG),  # + 256-token sequences, up to 3 epochs (all levers)
            ('bt300', BIG),            # + 3x the back-translations
            ('bt300-ffn64', BIG_FFN),  # + a larger adapter, also on the feed-forward layers
            # it<->en kept in stage 3 (E. Wdowiak's question): only stage 3 is retrained, on top of
            # the stage-2 adapter of bt300-ffn64-long, with 10,427 new WikiMatrix it-en pairs per
            # direction (as many as the scn-it pairs of the curated set)
            ('bt300-ffn64-long-iten', dict(LONG, stage3_iten=10427, stage2_from='bt300-ffn64-long')),
            # final model, all the data: (iv) plus the WikiMatrix it-scn training pairs in stage 3, with up
            # to four epochs (the validation score of (iv) still rose at the third); epoch chosen on validation
            ('final', dict(LONG, stage3_itscn=True, stage3_epochs=4, stage2_from='bt300-ffn64-long')),
            # ('bt300-ffn64-long-3.3B', dict(LONG, model_id='facebook/nllb-200-3.3B',
            #                                batch_size=8, grad_accum=2)),
            ]
for name, cfg in VARIANTS:
    staged.run(name, OUT, DATA, **cfg)

print(f"{'variant':24s} {'epoch':>5s} {'scn->en':>14s} {'en->scn':>14s}")
full = json.load(open(f'{OUT}/results_full.json'))
print(f"{'§7.2 (results_full)':24s} {'2':>5s} {str(full['scn->en']):>14s} {str(full['en->scn']):>14s}")
for name, _ in VARIANTS:
    p = f'{OUT}/results_{name}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        print(f"{name:24s} {r['epoch']:>5d} {str(r['test']['scn->en']):>14s} {str(r['test']['en->scn']):>14s}")

base: already done
bt300-ffn64-long: already done
bt300: already done
bt300-ffn64: already done
bt300-ffn64-long-iten: already done


/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


trainable params: 94,371,840 || all params: 1,465,010,176 || trainable%: 6.4417


Map:   0%|          | 0/30068 [00:00<?, ? examples/s]

Map:   0%|          | 0/30068 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/9389 [00:00<?, ? examples/s]

Map:   0%|          | 0/9389 [00:00<?, ? examples/s]

torch 2.11.0+cu130, transformers 4.57.6, accelerate 1.15.0


Step,Training Loss
6300,1.042300
6400,1.123100
6500,1.101400
6600,1.127200
6700,1.127800
6800,1.107700
6900,1.099700
7000,1.103700
7100,1.097900
7200,1.132600


final epoch 1: valid {'scn->en': (31.64, 57.4), 'en->scn': (20.41, 52.07)}
final epoch 2: valid {'scn->en': (32.78, 58.21), 'en->scn': (22.27, 53.12)}
final epoch 3: valid {'scn->en': (33.42, 58.69), 'en->scn': (23.15, 53.72)}
final epoch 4: valid {'scn->en': (33.58, 58.77), 'en->scn': (23.83, 54.19)}
final: epoch 4 chosen on validation, test {'scn->en': (36.01, 60.7), 'en->scn': (26.34, 55.75)}
final: on the test of E. Wdowiak {'en->scn': {'raw': 43.21, 'tokenized': 49.25}, 'scn->en': {'raw': 58.6, 'tokenized': 59.34}, 'it->scn': {'raw': 54.56, 'tokenized': 61.19}, 'scn->it': {'raw': 59.69, 'tokenized': 60.3}, 'it->en': {'raw': 57.69, 'tokenized': 58.42}, 'en->it': {'raw': 48.2, 'tokenized': 48.77}}
variant                  epoch        scn->en        en->scn
§7.2 (results_full)          2  [32.4, 58.19] [21.72, 52.79]
base                         2 [32.82, 58.41] [21.52, 52.68]
bt300-ffn64-long             3  [35.4, 60.39] [25.09, 55.24]
bt300                        2 [33.34, 58.75] 

### 7.5 Italian bridge — one trilingual model
Fine-tune four directions jointly (scn↔en + it↔scn) into a single adapter, for one epoch on the
web set. Needs `itsc_{train,test}.{it,scn}` on Drive (frozen WikiMatrix split). Results to
`results_multi.json`.

In [58]:
# --- 7.5 trilingual adapter (scn<->en, it<->scn) on the web set ---
isc = {s: {l: read(f'{DATA}/itsc_{s}.{l}') for l in ('it', 'scn')} for s in ('train', 'test')}
for s in ('train', 'test'):
    isc[s]['scn'] = [normalize(x, 'std') for x in isc[s]['scn']]
MULTI = f'{OUT}/nllb-lora-multi-1.3B'
p = f'{OUT}/results_multi.json'
res = json.load(open(p)) if os.path.exists(p) else {}
if not any(k.startswith('ft ') for k in res):
    if os.path.exists(f'{MULTI}/adapter_config.json'):
        mft, tok = load_base('facebook/nllb-200-1.3B', adapter=MULTI)
    else:
        m, tok = load_base('facebook/nllb-200-1.3B'); mft = attach_lora(m)
        mft = finetune(mft, tok, build_dataset(tok, [
            (train['scn'], train['en'], 'scn', 'en'), (train['en'], train['scn'], 'en', 'scn'),
            (isc['train']['it'], isc['train']['scn'], 'it', 'scn'),
            (isc['train']['scn'], isc['train']['it'], 'scn', 'it')]), out_dir=MULTI, epochs=1)
    for s, t, te in [('scn', 'en', test), ('en', 'scn', test), ('it', 'scn', isc['test']), ('scn', 'it', isc['test'])]:
        b, c = score(translate(mft, tok, te[s], s, t), te[t])
        res[f'ft {s}->{t}'] = {'bleu': b, 'chrf': c}
    json.dump(res, open(p, 'w'), indent=2)
for k, v in res.items():
    if k.startswith('ft '):
        print(f"{k[3:]}:  BLEU={v['bleu']}  chrF={v['chrf']}")

scn->en:  BLEU=30.75  chrF=56.57
en->scn:  BLEU=17.49  chrF=48.78
it->scn:  BLEU=26.97  chrF=51.69
scn->it:  BLEU=43.47  chrF=59.79


### 7.6 The two missing directions — it↔en on FLORES-200
Wdowiak reports all **six** directions among {scn, en, it}; our per-pair test sets only cover
four. FLORES-200 devtest (1012 sentences, multi-parallel, includes `scn_Latn`) fills the gap:
we score it↔en — directions the trilingual adapter **never saw** in fine-tuning — for both the
zero-shot base and the adapter, which doubles as a catastrophic-forgetting check. Set
`ALL_SIX = True` to score every direction on FLORES for one fully comparable table (slow).

In [59]:
# --- 7.6 it<->en on FLORES-200, zero-shot and with the trilingual adapter ---
def flores(lang, split='devtest'):                      # multi-parallel, aligned by line
    p = f'flores200_dataset/{split}/{lang}.{split}'
    if not os.path.exists(p):
        os.system('wget -q https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz'
                  ' -O fl.tar.gz && tar -xzf fl.tar.gz')
    if os.path.exists(p):
        return read(p)
    from datasets import load_dataset                   # last resort: FLORES+ (gated, needs HF login)
    return [r['text'] for r in load_dataset('openlanguagedata/flores_plus', lang, split=split)]

from nllb_pipeline import LANG
fl = {l: flores(LANG[l]) for l in ('it', 'en', 'scn')}
p = f'{OUT}/results_flores.json'
if not os.path.exists(p):
    ALL_SIX = False
    pairs = ([(s, t) for s in ('scn', 'en', 'it') for t in ('scn', 'en', 'it') if s != t]
             if ALL_SIX else [('it', 'en'), ('en', 'it')])
    flores_res, m = {}, None
    for name, adapter in [('zero-shot', None), ('multi-ft', f'{OUT}/nllb-lora-multi-1.3B')]:
        m = None                                            # drop stale model BEFORE loading (T4 OOM)
        m, tok = load_base('facebook/nllb-200-1.3B', adapter=adapter)
        for s, t in pairs:
            flores_res[f'{name} {s}->{t}'] = score(translate(m, tok, fl[s], s, t), fl[t])
            print(f'{name}  {s}->{t}:  BLEU={flores_res[f"{name} {s}->{t}"][0]}  chrF={flores_res[f"{name} {s}->{t}"][1]}')
    json.dump(flores_res, open(p, 'w'), indent=2)
for k, (bl, ch) in json.load(open(p)).items():
    print(f'{k}:  BLEU={bl}  chrF={ch}')

zero-shot it->en:  BLEU=34.42  chrF=62.65
zero-shot en->it:  BLEU=29.42  chrF=58.63
multi-ft it->en:  BLEU=27.14  chrF=57.92
multi-ft en->it:  BLEU=28.77  chrF=58.07


### 7.7 Other languages to and from Sicilian
Does the trilingual adapter, fine-tuned only on scn↔en and it↔scn, already translate
*other* Romance languages to and from Sicilian? FLORES-200 also has Spanish, French and
Portuguese. This reuses `flores()` and the loader from §7.6, and
tests zero-shot transfer through the shared Romance space.

In [60]:
# --- 7.7 es/fr/pt <-> scn with the trilingual adapter (FLORES-200) ---
from nllb_pipeline import LANG
LANG.update({'es': 'spa_Latn', 'fr': 'fra_Latn', 'pt': 'por_Latn'})   # extra Romance probes
p = f'{OUT}/results_other_langs.json'
if not os.path.exists(p):
    ol = {l: flores(LANG[l]) for l in ('es', 'fr', 'pt', 'scn')}
    opairs = [('es','scn'),('fr','scn'),('pt','scn'),('scn','es'),('scn','fr'),('scn','pt')]
    other_res = {}
    m = None
    m, tok = load_base('facebook/nllb-200-1.3B', adapter=f'{OUT}/nllb-lora-multi-1.3B')
    for s, t in opairs:
        other_res[f'{s}->{t}'] = score(translate(m, tok, ol[s], s, t), ol[t])
        print(f'{s}->{t}:  BLEU={other_res[f"{s}->{t}"][0]}  chrF={other_res[f"{s}->{t}"][1]}')
    json.dump(other_res, open(p, 'w'), indent=2)
for k, (bl, ch) in json.load(open(p)).items():
    print(f'{k}:  BLEU={bl}  chrF={ch}')

/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'peft_version', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


es->scn:  BLEU=7.81  chrF=39.35
fr->scn:  BLEU=9.89  chrF=42.04
pt->scn:  BLEU=11.39  chrF=42.81
scn->es:  BLEU=19.13  chrF=47.47
scn->fr:  BLEU=30.05  chrF=56.04
scn->pt:  BLEU=27.36  chrF=54.37
es->scn:  BLEU=7.81  chrF=39.35
fr->scn:  BLEU=9.89  chrF=42.04
pt->scn:  BLEU=11.39  chrF=42.81
scn->es:  BLEU=19.13  chrF=47.47
scn->fr:  BLEU=30.05  chrF=56.04
scn->pt:  BLEU=27.36  chrF=54.37


### 7.8 Our test and the test of Wdowiak, in raw and tokenized space
Two differences separate our numbers from the published baseline of Wdowiak (25.1 en->scn,
29.1 scn->en). (i) He scored in the space of the Napizia tokenizer, i.e., with accents folded
and contractions undone on output and reference, while our NLLB rows are scored on raw text,
where every accent or apostrophe that differs costs BLEU; on the June bidirectional adapter the
tokenized space adds about 3 BLEU on en->scn and 0.5 on scn->en. (ii) He scored on his own
hand-selected test, the 121 trilingual lines from AS38-39 of the `validation` sheet of his data
(`finetune/valid.*`). This cell translates both test sets with the adapters of §5.3, §7.2 and
§7.3 and scores them in both spaces, on all 121 lines of his test and on the lines that appear
in none of the training data of that model. Decoding is deterministic up to fp16 arithmetic on
the GPU (a few hundredths of BLEU), so the raw scores on our test must match `results_full.json` and `results_curriculum.json` (the bidirectional adapter
on Drive is the one of the June run, whose score is in `results.json`). Translations go to `hyp_<name>_<dir>_1.3B.txt`,
results to `results_tokenized.json`.

In [61]:
# --- 7.8 our test and Wdowiak's, raw and tokenized ---
!git -C SicilianNMT pull -q
import importlib, staged; importlib.reload(staged)
from staged import tokenized_bleu, clean_lines, eryk_scores
BT, FT = f'{OUT}/eryk', f'{OUT}/finetune'
test = {'scn': [normalize(x, 'std') for x in read(f'{DATA}/test.scn')], 'en': read(f'{DATA}/test.en')}
eryk = {l: read(f'{FT}/valid.{l}') for l in ('scn', 'en', 'it')}
web = [f'{DATA}/train.scn', f'{DATA}/train.en']
curated = [f'{FT}/train.scn-en.scn', f'{FT}/train.scn-en.en', f'{FT}/train.scn-it.scn', f'{FT}/train.scn-it.it']
bts = [f'{BT}/{f}' for f in sorted(os.listdir(BT)) if f.endswith(('.txt', '.en', '.it'))]
MODELS = {'bidir': web, 'full': curated + bts, 'curriculum': curated + bts}
p = f'{OUT}/results_tokenized.json'
if not os.path.exists(p):
    res = {}
    for name, train_files in MODELS.items():
        m, tok = load_base('facebook/nllb-200-1.3B', adapter=f'{OUT}/nllb-lora-{name}-1.3B')
        r = {}
        for s, t in [('scn', 'en'), ('en', 'scn')]:
            hyp = translate(m, tok, test[s], s, t)
            open(f'{OUT}/hyp_{name}_{s}2{t}_1.3B.txt', 'w', encoding='utf-8').write('\n'.join(hyp) + '\n')
            r[f'our test {s}->{t}'] = {'raw': score(hyp, test[t])[0], 'tokenized': tokenized_bleu(hyp, test[t], t)}
        keep = clean_lines(eryk, train_files)
        r['eryk test, all 121'] = eryk_scores(m, tok, eryk)
        r[f'eryk test, clean {len(keep)}'] = eryk_scores(m, tok, eryk, keep)
        res[name] = r
        print(name); [print(f'  {k}: {v}') for k, v in r.items()]
    json.dump(res, open(p, 'w'), indent=2)
for name, r in json.load(open(p)).items():
    print(name); [print(f'  {k}: {v}') for k, v in r.items()]

bidir
  our test scn->en: {'raw': 31.3, 'tokenized': 31.8}
  our test en->scn: {'raw': 19.19, 'tokenized': 21.92}
  eryk test, all 121: {'en->scn': {'raw': 28.9, 'tokenized': 32.19}, 'scn->en': {'raw': 50.29, 'tokenized': 51.16}, 'it->scn': {'raw': 35.95, 'tokenized': 39.69}, 'scn->it': {'raw': 52.09, 'tokenized': 52.98}, 'it->en': {'raw': 55.49, 'tokenized': 56.59}, 'en->it': {'raw': 46.61, 'tokenized': 47.31}}
  eryk test, clean 107: {'en->scn': {'raw': 28.5, 'tokenized': 31.95}, 'scn->en': {'raw': 49.88, 'tokenized': 50.65}, 'it->scn': {'raw': 35.47, 'tokenized': 39.5}, 'scn->it': {'raw': 51.65, 'tokenized': 52.57}, 'it->en': {'raw': 54.73, 'tokenized': 55.63}, 'en->it': {'raw': 45.15, 'tokenized': 45.89}}
full
  our test scn->en: {'raw': 32.37, 'tokenized': 33.03}
  our test en->scn: {'raw': 21.72, 'tokenized': 25.69}
  eryk test, all 121: {'en->scn': {'raw': 38.9, 'tokenized': 44.4}, 'scn->en': {'raw': 56.66, 'tokenized': 57.0}, 'it->scn': {'raw': 55.14, 'tokenized': 61.04}, 'scn-

## 8. Results & discussion
All the numbers of the paper are printed by the cells above, each from the results file of the
model that produced it: the zero-shot rows (§5.2), the bidirectional adapter (§6), back-translation
(§7.1), the staged runs (§7.2, §7.3), the ablation and the final model chosen on validation
(§7.4), the Italian directions (§7.5–§7.7) and the scores on Wdowiak's test set in his tokenized
space (§7.8).

## 9. Serving
The saved adapter backs a local web app and a Telegram bot — one FastAPI `/translate`
endpoint over `experiments/serving/translator.py`. See `experiments/serving/README.md`:
`uvicorn api:app` serves the site at `/`, `telegram_bot.py` runs the bot, and a Docker +
compose deploy scaffold is in the same folder.

## 10. Conclusion
A reproducible, Perl-free path from the PDFs of *Arba Sicula* to a trilingual Sicilian translator:
NLLB-200 adapted with LoRA, trained in stages on Wdowiak's back-translations and on the curated
literary text, with the final model chosen on the validation set. The comparison with his
published systems, on his own test set and in his tokenized space, is in §7.8 and in the paper.